# 01 · Interactions EDA

Volume over time, how heavy-tailed users and games are, cold start, and how users' history
relates to their next review. Full-data aggregates are streamed and cached; per-user analyses use
a user sample (`SAMPLE_USERS`, whole histories).

In [ ]:
import os

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from steam_eda import (
    Sample,
    binned_rate,
    cached,
    categorical_rate,
    decode,
    feature_report,
    game_catalog,
    gini,
    iter_mart,
    load_mart,
    load_raw,
    lookup,
    lorenz,
    mart_schema,
    mart_table,
    raw_files,
    review_counts,
    tag_profile_similarity,
    with_user_history,
)
from steam_eda.plots import loglog_hist, rate_plot, setup

setup()

SAMPLE_USERS = float(os.environ.get("EDA_SAMPLE_USERS", 0.02))  # 2% of users ≈ 2M rows
VALIDATION_FRACTION = 0.1  # training's temporal split (TrainingSettings.validation_fraction)

## Volume over time (all interactions)

In [ ]:
def monthly() -> pl.DataFrame:
    parts = [
        b.group_by(month=pl.col("timestamp").dt.truncate("1mo")).agg(
            reviews=pl.len(), positives=pl.col("is_positive").sum()
        )
        for b in iter_mart("interactions", columns=["timestamp", "is_positive"])
    ]
    return (
        pl.concat(parts).group_by("month").agg(pl.col("reviews", "positives").sum()).sort("month")
    )


months = cached("monthly_interactions", monthly).with_columns(
    positive_rate=pl.col("positives") / pl.col("reviews"),
    cumulative_share=pl.col("reviews").cum_sum() / pl.col("reviews").sum(),
)
cutoff_month = months.filter(pl.col("cumulative_share") >= 1 - VALIDATION_FRACTION)["month"][0]

fig, (a, b) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
data = months.filter(pl.col("month") >= pl.datetime(2010, 1, 1)).to_pandas()
a.bar(data["month"], data["reviews"], width=25)
a.set_yscale("log")
a.set_title("reviews per month (log)")
b.plot(data["month"], data["positive_rate"])
b.set_title("positive rate per month")
for ax in (a, b):
    ax.axvline(cutoff_month, color="red", ls="--", label="≈ validation cutoff")
a.legend()
print(
    f"{months['reviews'].sum():,d} interactions, positive rate "
    f"{months['positives'].sum() / months['reviews'].sum():.3f}, cutoff ≈ {cutoff_month:%Y-%m}"
)

## User sample

In [ ]:
inter = load_mart(
    "interactions",
    columns=["user_id", "game_idx", "timestamp", "is_positive"],
    sample=Sample.by_key("user_id", SAMPLE_USERS),
)
inter = with_user_history(inter)
per_user = inter.group_by("user_id").agg(
    reviews=pl.len(),
    positives=pl.col("is_positive").sum(),
    first=pl.col("timestamp").min(),
    last=pl.col("timestamp").max(),
)
print(f"{inter.height:,d} interactions of {per_user.height:,d} users")
per_user.select(
    pl.col("reviews").mean().alias("mean reviews/user"),
    pl.col("reviews").median().alias("median"),
    (pl.col("reviews") == 1).mean().alias("share with 1 review"),
    (pl.col("positives") >= 5).mean().alias("share with ≥5 positives (full user-tower history)"),
)

## Long tails: users and games

In [ ]:
counts = cached("review_counts", review_counts)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
loglog_hist(per_user["reviews"], "reviews per user (sample)", axes[0])
loglog_hist(counts["reviews"], "reviews per game (all)", axes[1])
for name, series in {"users": per_user["reviews"], "games": counts["reviews"]}.items():
    curve = lorenz(series).to_pandas()
    axes[2].plot(
        curve["top_share"], curve["share_of_total"], label=f"{name} (gini {gini(series):.2f})"
    )
axes[2].set_xscale("log")
axes[2].set_title("share of reviews held by the top x")
axes[2].legend()

catalog_size = cached("catalog", game_catalog).height
top = counts.sort("reviews", descending=True)
print(
    f"{counts.height:,d} of {catalog_size:,d} catalog games have reviews; "
    f"top 1% of games hold {top.head(counts.height // 100)['reviews'].sum() / top['reviews'].sum():.1%}"
)

## Cold start

The user tower needs history: a review with no earlier positive is a cold row (training keeps
only `COLD_ROW_FRACTION` of them; inference never sees them).

In [ ]:
positives = inter.filter(pl.col("is_positive"))
print(f"cold share of positive reviews: {(positives['prior_positives'] == 0).mean():.1%}")
history = (
    positives.with_columns(bucket=pl.col("prior_positives").clip(upper_bound=20))
    .group_by("bucket")
    .len()
    .sort("bucket")
)
ax = history.to_pandas().plot.bar(x="bucket", y="len", legend=False, figsize=(12, 3))
ax.set_title("positive reviews by the user's earlier positives (20 = 20+)")

## Does history predict sentiment?

A user's earlier positive rate vs the label of the next review. A strong curve means a user bias
term (or the rate as a ranker feature) is worth having.

In [ ]:
warm = inter.filter(pl.col("prior_reviews") >= 3)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rate_plot(
    binned_rate(warm, "prior_positive_rate", bins=10),
    "rate vs user's prior positive rate",
    ax=axes[0],
)
rate_plot(binned_rate(inter, "prior_reviews", bins=10), "rate vs user's prior reviews", ax=axes[1])
rate_plot(
    binned_rate(inter, "days_since_prior_review", bins=10),
    "rate vs days since prior review",
    ax=axes[2],
)
axes[1].set_xscale("symlog")
axes[2].set_xscale("symlog")

## Users are different raters

In [ ]:
raters = per_user.filter(pl.col("reviews") >= 10).with_columns(
    rate=pl.col("positives") / pl.col("reviews")
)
ax = sns.histplot(raters["rate"].to_pandas(), bins=40)
ax.set_title(f"positive rate of users with ≥10 reviews (n={raters.height:,d})")

## Time between reviews

In [ ]:
gaps = inter.filter(pl.col("days_since_prior_review") > 0)["days_since_prior_review"]
ax = sns.histplot(gaps.log10().to_pandas(), bins=60)
ax.set_xlabel("log10 days since the user's previous review")
print(gaps.describe())

## Notes

Write what you found here (volume trend, cutoff month, tail shares, cold share, rater bias), and
what it means for sampling, the split and features.